<a href="https://colab.research.google.com/github/ArshnoorSinghh/ML-Project/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 (paper, #1 "Anatomy of Growing Content"): growing pages average
37.6% more words and are 20% younger than declining pages; the paper suggests
expanding thin pages that already earn impressions.

My methodology question — where does the label come from? "Growing" vs
"declining" is defined from a 30-day-vs-previous-30-day impression change. I'd
ask how sensitive the finding is to that window: would a 90-day trend label the
same pages the same way? And since the comparison is observational, the
length/age gap is a correlation — a third factor (topic or intent) could drive
both length and growth, so I'd want to know whether expanding a thin page
actually causes growth or whether length just marks already-strong pages.

Finding 2 (paper, #3 "Click Capture by Position Tier"): weighted CTR falls from
0.423% in the top-3 to 0.050% deep, an 88% drop; the paper recommends refining
snippets on page-one pages.

My methodology question — does the validation design support the claim? The
CTRs are portfolio-level weighted averages (total clicks / total impressions
per tier). A weighted average can be dominated by a few very high-impression
pages or large brands, so I'd ask whether the tier pattern holds within
individual brands too — a per-brand (grouped) check, like the client-holdout I
used in my own model, would confirm it isn't a brand-composition effect. I'd
also note the "refine snippets -> click lift" step is associational, since the
paper observes the tier gap without testing an intervention.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before/after split comparison:
- Naive random split R2 = 0.076
- Grouped-by-client split R2 = 0.106

Normally a naive random split inflates the score, because a client's pages can
appear in both train and test and the model leans on client-specific patterns.
Here the two numbers are close (and the naive one is actually slightly lower),
which is itself informative: my model is weak (R2 ~0.10 on noisy data with only
7 engagement features), so there is little client-specific signal for a random
split to exploit. The gap between the two splits is within the noise from which
rows land in the test set. I report the grouped R2 (0.106) as my honest estimate,
since it is the design that guarantees the test clients are genuinely unseen.

In [7]:
import pandas as pd, numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ---- load March + rebuild features (same as ML-08) ----
HF_TOKEN = userdata.get('HF_TOKEN')
url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03"
df = pd.read_parquet(url, storage_options={"token": HF_TOKEN})

d = df[df["gsc_data_available"] == True].copy()
d["ctr"] = d["gsc_clicks"] / d["gsc_impressions"]
d["avg_position"] = d["gsc_sum_position"] / d["gsc_impressions"]
d = d[d["gsc_impressions"] >= 100].copy()
d["position_tier"] = np.select(
    [d["avg_position"] <= 3, d["avg_position"] <= 10, d["avg_position"] <= 20],
    ["top_3","page_1","page_2"], default="deep")

base = d[d["ga4_data_available"] == True].copy()
features = ["sessions_organic","ga4_pageviews","ga4_engaged_sessions",
            "ga4_sessions","scroll_events","ga4_users","ga4_total_engagement_sec"]

# ---- helper: build target train-only, fit, score ----
def build_and_score(train_rows, test_rows):
    tr = base.iloc[train_rows].copy()
    te = base.iloc[test_rows].copy()
    tier_means = tr.groupby("position_tier")["ctr"].mean()       # train-only means
    tr["gap"] = tr["position_tier"].map(tier_means) - tr["ctr"]
    te["gap"] = te["position_tier"].map(tier_means) - te["ctr"]
    sc = StandardScaler()
    Xtr = sc.fit_transform(tr[features]); Xte = sc.transform(te[features])
    m = LinearRegression().fit(Xtr, tr["gap"])
    return r2_score(te["gap"], m.predict(Xte))

# ---- BEFORE: naive random split (clients can leak across) ----
tr_idx, te_idx = train_test_split(np.arange(len(base)), test_size=0.2, random_state=42)
r2_naive = build_and_score(tr_idx, te_idx)

# ---- AFTER: grouped split by client (honest) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
gtr_idx, gte_idx = next(gss.split(base, groups=base["client_hash_id"]))
r2_grouped = build_and_score(gtr_idx, gte_idx)

print(f"BEFORE — naive random split   R2 = {r2_naive:.4f}")
print(f"AFTER  — grouped by client    R2 = {r2_grouped:.4f}")

BEFORE — naive random split   R2 = 0.0762
AFTER  — grouped by client    R2 = 0.1055


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit — my final feature set:

Features used: the 7 GA4 engagement columns (sessions_organic, ga4_pageviews,
ga4_engaged_sessions, ga4_sessions, scroll_events, ga4_users,
ga4_total_engagement_sec).

None of these is used to compute the target. The target (CTR gap) is built from
GSC search data (clicks, impressions, position). GA4 measures on-site behaviour
AFTER a click, so it is a separate measurement system from the search-side CTR
being predicted — using it is not circular.

Deliberately excluded as leaky:
- ctr and gsc_clicks: the target is built from these, so including them would
  let the model read the answer (I demonstrated this in my Week-3 leakage trap).
- expected_ctr and ctr_gap: these ARE the target.
- avg_position / position_tier: the target is defined relative to position
  tiers, so using position as a feature would be circular.

Check: the correlation of every final feature with the target is modest
(roughly 0.1 to 0.36 in magnitude), none near +/-1. A feature correlating near
+/-1 would be a red flag for leakage; the honest, modest correlations here are
consistent with a real (non-leaky) signal.

One caveat: tier means for the target were computed on training data only, so
the target itself does not leak test information into training.


In [8]:
# Leakage audit: confirm my features don't contain the answer.
# Show correlation of each FINAL feature with the target — none should be near ±1.
audit = base.copy()
tier_means = audit.groupby("position_tier")["ctr"].mean()
audit["ctr_gap"] = audit["position_tier"].map(tier_means) - audit["ctr"]

print("Correlation of each FINAL feature with the target (ctr_gap):")
print(audit[features + ["ctr_gap"]].corr()["ctr_gap"].drop("ctr_gap").round(3))

print("\nExcluded (leaky) columns and why they were dropped:")
print("- ctr, gsc_clicks: used to BUILD the target -> would be circular")
print("- expected_ctr, ctr_gap: are the target itself")
print("- avg_position / position_tier: define the target's tiers -> circular")

Correlation of each FINAL feature with the target (ctr_gap):
sessions_organic           -0.277
ga4_pageviews              -0.083
ga4_engaged_sessions       -0.117
ga4_sessions               -0.063
scroll_events              -0.081
ga4_users                  -0.057
ga4_total_engagement_sec   -0.088
Name: ctr_gap, dtype: float64

Excluded (leaky) columns and why they were dropped:
- ctr, gsc_clicks: used to BUILD the target -> would be circular
- expected_ctr, ctr_gap: are the target itself
- avg_position / position_tier: define the target's tiers -> circular


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Claim rewrite:

Bold version (too strong):
"My model identifies underperforming pages, and rewriting their titles will
recover the missing clicks."

Honest rewrite:
"My model produces a decision-support ranking: it orders pages by their
observed association between low on-site engagement and a larger measured CTR
gap for their position tier. This is directional evidence that flags pages
worth a human review, not proof that any page under-earns for a fixable reason
or that a rewrite will recover clicks. The relationship is observational
(correlation, not a tested intervention), the model explains only ~10% of the
variance, and it was validated on a small number of held-out clients, so the
estimate is noisy. The output is a prioritised review queue, not an outcome
guarantee."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.